# 03 — Kotha et al. (2026) "DiaBeats": random forest, single split vs cross-validation

**Method (IEEE I3CTCON):** a random forest on the raw clinical features, reported from one
80/20 test split (Pima). Here the same model runs on all 4 diabetes datasets.

**Protocols**
* `paper` — one 80/20 stratified split (seed 42), ungrouped, no scaling (RF on raw features;
  NaNs median-imputed on all rows).
* `paper_10splits` — the same single-split recipe repeated with 10 different split seeds, to show
  how much a single-split number can move.
* `clean` — grouped/stratified 5-fold × seeds {0, 1, 42}, in-fold imputation.

A threshold-tuned variant (threshold chosen on the training fold by inner CV to maximise
balanced accuracy) is added in `clean` only.

In [1]:
import os, sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd() / "notebooks2", Path("/content/notebooks2"), *Path("/kaggle/input").glob("*"), *Path("/kaggle/input").glob("*/notebooks2")]:
    if (p / "nb2_common.py").exists():
        sys.path.insert(0, str(p)); break
from nb2_common import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
sanity_check_kernels()
DATA = load_all()

nb2_common loaded: numpy 2.4.6, sklearn 1.6.1
kernel self-test passed
pima     N=  768 d=  8 prevalence=0.349 unique=  768 grouped_cv=False  [diabetes]
sylhet   N=  520 d= 16 prevalence=0.615 unique=  251 grouped_cv=True  [diabetes (early stage)]


cdc      N= 2000 d= 21 prevalence=0.140 unique= 1995 grouped_cv=True  [diabetes or prediabetes (self-reported)]


hosp130  N= 2000 d= 44 prevalence=0.088 unique= 2000 grouped_cv=False  [readmitted within 30 days]


In [2]:
def impute_only(X_fit, *others):
    imp = SimpleImputer(strategy="median").fit(X_fit)
    return [imp.transform(X_fit)] + [imp.transform(o) for o in others]


def make_method(tune_threshold):
    def kotha_method(Xtr, ytr, Xte, seed, groups_tr, X_prep):
        Ar, Ae = prep(impute_only, X_prep, Xtr, Xte)
        rf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1).fit(Ar, ytr)
        p = rf.predict_proba(Ae)[:, 1]
        out = {"DiaBeats RF": ((p >= 0.5).astype(int), p)}
        if tune_threshold:
            folds = inner_splits(ytr, groups_tr, seed, n_splits=3)
            oof = cross_val_predict(RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
                                    Ar, ytr, cv=folds, method="predict_proba")[:, 1]
            grid = np.linspace(0.05, 0.95, 19)
            t = grid[np.argmax([balanced_accuracy_score(ytr, (oof >= g).astype(int)) for g in grid])]
            out["DiaBeats RF (threshold by inner CV)"] = ((p >= t).astype(int), p)
        Sr, Se = prep(impute_scale, X_prep, Xtr, Xte)
        lr = LogisticRegression(max_iter=1000).fit(Sr, ytr)
        out["Logistic Regression"] = (lr.predict(Se), lr.predict_proba(Se)[:, 1])
        return out
    return kotha_method


rec = Recorder("P03_Kotha", "03_kotha", config=dict(
    method="RF n_estimators=100 on raw features; LR reference; inner-CV threshold variant (clean only)",
    paper_protocol="single 80/20 stratified split seed 42; paper_10splits = seeds 0..9"))
print("paper protocol"); run_paper_holdout(rec, DATA, make_method(False), test_size=0.2, global_prep=True)
for s in range(10):
    run_paper_holdout(rec, DATA, make_method(False), test_size=0.2, global_prep=True, seed=s,
                      protocol="paper_10splits")
print("clean protocol"); run_clean(rec, DATA, make_method(True))
df = rec.save()

paper protocol


  paper  pima     test rows sharing a feature vector with train: 0


  paper  sylhet   test rows sharing a feature vector with train: 72


  paper  cdc      test rows sharing a feature vector with train: 1


  paper  hosp130  test rows sharing a feature vector with train: 0


  paper_10splits pima     test rows sharing a feature vector with train: 0


... (39 outputs left out)

  clean  pima     done in   18.3s


  clean  sylhet   done in   17.6s


  clean  cdc      done in   27.0s


  clean  hosp130  done in   23.2s
saved /content/notebooks2/results/03_kotha.csv  (268 rows)


In [3]:
pd.set_option("display.width", 220, "display.max_rows", 200)
summarise(df)

,dataset,protocol,model,accuracy,balanced_accuracy,recall,specificity,roc_auc,n_runs
0,cdc,clean,DiaBeats RF,0.863 ± 0.015,0.559 ± 0.015,0.138 ± 0.032,0.981 ± 0.007,0.796 ± 0.025,15
1,cdc,clean,DiaBeats RF (threshold by inner CV),0.701 ± 0.054,0.721 ± 0.031,0.748 ± 0.083,0.694 ± 0.072,0.796 ± 0.025,15
2,cdc,clean,Logistic Regression,0.863 ± 0.017,0.577 ± 0.018,0.18 ± 0.039,0.974 ± 0.006,0.818 ± 0.02,15
3,cdc,paper,DiaBeats RF,0.875 ± nan,0.569 ± nan,0.143 ± nan,0.994 ± nan,0.823 ± nan,1
4,cdc,paper,Logistic Regression,0.865 ± nan,0.563 ± nan,0.143 ± nan,0.983 ± nan,0.824 ± nan,1
5,cdc,paper_10splits,DiaBeats RF,0.862 ± 0.009,0.563 ± 0.023,0.148 ± 0.045,0.979 ± 0.008,0.799 ± 0.025,10
6,cdc,paper_10splits,Logistic Regression,0.859 ± 0.008,0.569 ± 0.029,0.166 ± 0.062,0.972 ± 0.008,0.81 ± 0.022,10
7,hosp130,clean,DiaBeats RF,0.912 ± 0.001,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.001,0.572 ± 0.048,15
8,hosp130,clean,DiaBeats RF (threshold by inner CV),0.622 ± 0.197,0.538 ± 0.033,0.437 ± 0.252,0.64 ± 0.24,0.572 ± 0.048,15
9,hosp130,clean,Logistic Regression,0.909 ± 0.004,0.503 ± 0.008,0.009 ± 0.014,0.996 ± 0.004,0.6 ± 0.05,15


## Spread of the single-split accuracy across 10 split seeds (min / max)

In [4]:
df[df.protocol == "paper_10splits"].groupby(["dataset", "model"])["accuracy"].agg(["min", "max", "mean", "std"]).round(3)

min    max   mean    std
dataset model                                          
cdc     DiaBeats RF          0.850  0.880  0.862  0.009
        Logistic Regression  0.845  0.868  0.859  0.008
hosp130 DiaBeats RF          0.912  0.912  0.912  0.000
        Logistic Regression  0.902  0.912  0.909  0.003
pima    DiaBeats RF          0.695  0.812  0.750  0.038
        Logistic Regression  0.682  0.792  0.756  0.035
sylhet  DiaBeats RF          0.952  1.000  0.983  0.015
        Logistic Regression  0.875  0.962  0.922  0.032